# NINAAD WAGLE | BTECH AI SEM V | I065 B2 | NLP LAB 7

Word Sense Disambiguation (WSD) means picking the correct meaning of a word that has more than one
meaning. The word *bank* can be a place that keeps money or the land next to a river. Only the rest
of the sentence tells us which one it is.

The dataset gives us sentences, a polysemy word in each sentence, and the meaning that word carries
there. Every row in the file is a correct pairing, so there are no wrong examples to learn from.
Because of that we do not build a "correct / incorrect" checker. Instead we build a model that is
given a sentence and the word, and has to pick the right meaning out of that word's known meanings.
If the model picks the meaning that was supplied, that pairing was right.

# Part A : To load the WSD dataset and list the senses of each word

## Step 1 : Reading the Excel file and collecting every meaning a word can have

`pd.read_excel` reads one sheet of the spreadsheet into a table. The `Combined` sheet has one row per
sentence. We group the table by the polysemy word and collect the distinct meanings written against
it, which gives us the list of choices the model will have to pick from for that word.

Words that appear with only one meaning are dropped, because there is nothing to disambiguate there
and keeping them would make the score look better than it really is.

In [1]:
import re, math
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import wordnet as wn, stopwords
from nltk.wsd import lesk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

for pkg in ['wordnet', 'omw-1.4', 'stopwords']:
    nltk.download(pkg, quiet=True)

df = pd.read_excel('WSD_test_data_combined.xlsx', sheet_name='Combined')
df.columns = ['sn', 'sentence', 'word', 'meaning']
df['word'] = df['word'].str.strip().str.lower()
df['meaning'] = df['meaning'].str.strip()

# every distinct meaning seen for each word, keeping only the words that really are ambiguous
senses = {w: list(m) for w, m in df.groupby('word')['meaning'].unique().items() if len(m) > 1}
df = df[df['word'].isin(senses)].reset_index(drop=True)

print('rows kept        :', len(df))
print('ambiguous words  :', len(senses))
print('senses of "bank" :', senses['bank'])
df.head()

rows kept        : 4006
ambiguous words  : 300
senses of "bank" : ['financial institution that holds money and gives loans', 'land alongside a river, lake or sea']


,sn,sentence,word,meaning
0,1,I have bank account.,bank,financial institution that holds money and giv...
1,2,Loan amount is approved by the bank.,bank,financial institution that holds money and giv...
2,3,He returned to office after he deposited cash ...,bank,financial institution that holds money and giv...
3,4,They started using new software in their bank.,bank,financial institution that holds money and giv...
4,5,he went to bank balance inquiry.,bank,financial institution that holds money and giv...


# Part B : To pick the sense by word overlap (Lesk), without any training

## Step 1 : Simplified Lesk written manually

The Lesk idea is very simple: the correct meaning is the one whose wording shares the most words
with the sentence. For *"he is walking along the river bank"* the meaning *"land alongside a river,
lake or sea"* shares the word **river** with the sentence, while *"financial institution that holds
money"* shares nothing, so Lesk picks the first one.

`toks` is our small cleaner. It lowercases the text, keeps only letters, throws away stopwords
(*the, is, of* and the like, which match everything and mean nothing) and throws away very short
words. The target word itself is removed too, since it sits in every candidate and cannot separate
them.

The data is split into a training part and a test part now. Lesk does not use the training part, but
every method below is scored on the same test sentences so the numbers can be compared fairly.

In [2]:
STOP = set(stopwords.words('english'))

def toks(text):
    return [w for w in re.findall(r"[a-z]+", str(text).lower()) if w not in STOP and len(w) > 2]

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

def lesk_manual(sentence, word):
    context = set(toks(sentence)) - {word}
    return max(senses[word], key=lambda m: len(context & set(toks(m))))

def accuracy(predict):
    return np.mean([predict(r.sentence, r.word) == r.meaning for r in test_df.itertuples()])

print('train / test sentences :', len(train_df), '/', len(test_df))
print('manual Lesk accuracy   :', round(accuracy(lesk_manual), 3))
print()
print(lesk_manual('he is waking along the river bank.', 'bank'))
print(lesk_manual('He took loan from a Bank.', 'bank'))

train / test sentences : 3204 / 802
manual Lesk accuracy   : 0.602

land alongside a river, lake or sea
financial institution that holds money and gives loans


## Step 2 : Lesk using NLTK's inbuilt function

`nltk.wsd.lesk` does the same overlap trick but against **WordNet**, a dictionary built into NLTK.
WordNet stores words in groups called *synsets*, and each synset has a short definition. `lesk`
returns the synset whose definition overlaps the sentence most.

That synset is WordNet's own wording, not the wording used in our file, so we still have to map it
back: we keep the meaning from our dataset whose text shares the most words with the chosen synset's
definition. The extra hop loses information, which is why this scores lower than our own version.

In [3]:
def lesk_nltk(sentence, word):
    syn = lesk(toks(sentence), word)
    if syn is None:                       # WordNet does not know this word
        return senses[word][0]
    described = set(toks(syn.definition())) | {l.name().lower() for l in syn.lemmas()}
    return max(senses[word], key=lambda m: len(described & set(toks(m))))

print('NLTK lesk accuracy :', round(accuracy(lesk_nltk), 3))
print()
for s in wn.synsets('bank')[:3]:
    print(s.name(), '->', s.definition())

NLTK lesk accuracy : 0.512

bank.n.01 -> sloping land (especially the slope beside a body of water)
depository_financial_institution.n.01 -> a financial institution that accepts deposits and channels the money into lending activities
bank.n.03 -> a long ridge or pile


# Part C : To pick the sense by training a Naive Bayes model on the example sentences

## Step 1 : Naive Bayes written manually

Lesk only looks at the dictionary wording, so it misses that *loan*, *deposit* and *account* point at
the money sense even though none of those words appear in the definition. Training fixes this: the
dataset already contains several example sentences for each sense, so we can simply learn which
words tend to show up around each sense.

Naive Bayes is the simplest way to do that. For every sense we count how often each word appeared
near it in the training sentences. To score a new sentence we add up, for each of its words, how
likely that word is under that sense, and pick the sense with the highest total. It is called
*naive* because it treats the words as independent of one another.

Two small details. `+1` is added to every count (add-one smoothing) so that a word never seen with a
sense gives a very small probability instead of zero, which would wipe out the whole score. And
`log(number of training sentences + 1)` is added as the prior, which mildly favours the sense that is
simply more common.

In [4]:
counts = defaultdict(Counter)     # (word, meaning) -> how often each context word appeared
n_rows = Counter()                # (word, meaning) -> how many training sentences

for r in train_df.itertuples():
    counts[(r.word, r.meaning)].update(x for x in toks(r.sentence) if x != r.word)
    n_rows[(r.word, r.meaning)] += 1

for w, meanings in senses.items():          # the meaning's own wording counts as evidence too
    for m in meanings:
        counts[(w, m)].update(toks(m))

V = len({x for c in counts.values() for x in c})       # vocabulary size
total = {k: sum(c.values()) for k, c in counts.items()}

def nb_manual(sentence, word):
    context = [x for x in toks(sentence) if x != word]
    def score(m):
        c, t = counts[(word, m)], total[(word, m)]
        return math.log(n_rows[(word, m)] + 1) + sum(math.log((c[x] + 1) / (t + V)) for x in context)
    return max(senses[word], key=score)

print('manual Naive Bayes accuracy :', round(accuracy(nb_manual), 3))
print()
print(nb_manual('I need to deposit my salary there.', 'bank'))
print(nb_manual('We sat quietly near the river.', 'bank'))

manual Naive Bayes accuracy : 0.708

financial institution that holds money and gives loans
land alongside a river, lake or sea


## Step 2 : The same model with scikit-learn, checked against the manual one

`MultinomialNB` is scikit-learn's version of exactly the counting above, and `CountVectorizer` turns
the sentences into the word-count table it needs. Each `(word, meaning)` pair is given a number so it
can act as a class label, and one extra training row per sense carries the meaning's own wording,
matching what we did by hand.

The model is trained over all senses at once, so when predicting we look only at the columns
belonging to the senses that the target word actually has, and take the best of those. The two
versions agree on almost every test sentence, which confirms the manual code was right.

In [5]:
sense_id = {k: i for i, k in enumerate(sorted({(w, m) for w in senses for m in senses[w]}))}
id_sense = {i: k for k, i in sense_id.items()}

def context_text(sentence, word):
    return ' '.join(x for x in toks(sentence) if x != word)

docs = [context_text(r.sentence, r.word) for r in train_df.itertuples()]
ys   = [sense_id[(r.word, r.meaning)] for r in train_df.itertuples()]
docs += [' '.join(toks(m)) for (w, m) in sense_id]        # the meaning wording, as before
ys   += [sense_id[(w, m)] for (w, m) in sense_id]

cv = CountVectorizer(tokenizer=toks, token_pattern=None)
nb = MultinomialNB(alpha=1.0).fit(cv.fit_transform(docs), ys)

def nb_sklearn(sentence, word):
    scores = nb.predict_log_proba(cv.transform([context_text(sentence, word)]))[0]
    allowed = [sense_id[(word, m)] for m in senses[word]]
    return id_sense[allowed[int(np.argmax(scores[allowed]))]][1]

manual_pred  = [nb_manual(r.sentence, r.word) for r in test_df.itertuples()]
sklearn_pred = [nb_sklearn(r.sentence, r.word) for r in test_df.itertuples()]

print('sklearn Naive Bayes accuracy :', round(accuracy(nb_sklearn), 3))
print('agreement with manual model  :', round(np.mean([a == b for a, b in zip(manual_pred, sklearn_pred)]), 3))

sklearn Naive Bayes accuracy : 0.707
agreement with manual model  : 0.994


## Step 3 : Giving each sense extra words from WordNet

Some senses have only two or three example sentences, which is too little to learn from. WordNet can
top them up for free. For each meaning we find the synset whose definition looks most like our
meaning text, then add that synset's definition and example sentences to the sense's training words.

So the money sense of *bank* also learns *deposits*, *lending* and *activities* from WordNet, even
though no training sentence used those words. This is the final model.

In [6]:
def wordnet_words(meaning, word):
    text = set(toks(meaning))
    best, best_overlap = None, 0
    for s in wn.synsets(word):                  # the synset that matches our meaning text most
        overlap = len(text & set(toks(s.definition())))
        if overlap > best_overlap:
            best, best_overlap = s, overlap
    if best is None:
        return []
    return toks(best.definition()) + [t for e in best.examples() for t in toks(e)]

docs_wn = [context_text(r.sentence, r.word) for r in train_df.itertuples()]
ys_wn   = [sense_id[(r.word, r.meaning)] for r in train_df.itertuples()]
docs_wn += [' '.join(toks(m) + wordnet_words(m, w)) for (w, m) in sense_id]
ys_wn   += [sense_id[(w, m)] for (w, m) in sense_id]

cv_wn = CountVectorizer(tokenizer=toks, token_pattern=None)
nb_wn = MultinomialNB(alpha=1.0).fit(cv_wn.fit_transform(docs_wn), ys_wn)

def final_model(sentence, word):
    scores = nb_wn.predict_log_proba(cv_wn.transform([context_text(sentence, word)]))[0]
    allowed = [sense_id[(word, m)] for m in senses[word]]
    return id_sense[allowed[int(np.argmax(scores[allowed]))]][1]

print('final model accuracy :', round(accuracy(final_model), 3))
print()
print('words WordNet adds to the money sense of bank:')
print(wordnet_words(senses['bank'][0], 'bank'))

final model accuracy : 0.726

words WordNet adds to the money sense of bank:
['financial', 'institution', 'accepts', 'deposits', 'channels', 'money', 'lending', 'activities', 'cashed', 'check', 'bank', 'bank', 'holds', 'mortgage', 'home']


# Part D : To compare the methods and try the final model

## Step 1 : Accuracy of every method side by side

The *most frequent sense* row is the lazy baseline: always answer with whichever meaning of that word
appeared most often in training, ignoring the sentence completely. Any real method has to beat it to
be worth anything.

In [7]:
def most_frequent(sentence, word):
    return max(senses[word], key=lambda m: n_rows[(word, m)])

results = pd.DataFrame({
    'method': ['most frequent sense (baseline)', 'manual Lesk', 'NLTK lesk',
               'manual Naive Bayes', 'sklearn Naive Bayes', 'Naive Bayes + WordNet (final)'],
    'accuracy': [round(accuracy(f), 3) for f in
                 [most_frequent, lesk_manual, lesk_nltk, nb_manual, nb_sklearn, final_model]]
})
results

,method,accuracy
0,most frequent sense (baseline),0.514
1,manual Lesk,0.602
2,NLTK lesk,0.512
3,manual Naive Bayes,0.708
4,sklearn Naive Bayes,0.707
5,Naive Bayes + WordNet (final),0.726


## Step 2 : Looking at what the final model predicted

A few test sentences with the meaning the model chose next to the meaning the dataset gives. This
shows where it still slips: short sentences with no helpful words around the target are the usual
failures, because there is simply nothing for the model to go on.

In [8]:
sample = test_df.sample(8, random_state=1).copy()
sample['predicted'] = [final_model(r.sentence, r.word) for r in sample.itertuples()]
sample['correct'] = sample['predicted'] == sample['meaning']
sample[['sentence', 'word', 'meaning', 'predicted', 'correct']]

,sentence,word,meaning,predicted,correct
2004,He had a scruffy old teddy bear which went by ...,bear,large heavy mammal (incl. teddy bear),large heavy mammal (incl. teddy bear),True
2654,Concrete block is used in the wall,block,"solid piece of hard material (wood, stone, ice...","solid piece of hard material (wood, stone, ice...",True
1602,They are watching a game of chess.,game,"activity or contest with rules, a match","activity or contest with rules, a match",True
3052,"In its ruling , the state Board of Education u...",board,group of people who govern an organisation,group of people who govern an organisation,True
817,The old man was carrying a load of stick.,stick,thin piece of wood,"to adhere, attach or stay together",False
240,You got negative value.,negative,less than zero,pessimistic or unfavourable,False
3873,"Before hitting the pole , Mr. Stone s car brus...",left,direction or side opposite to right,direction or side opposite to right,True
2628,There was a line of people waiting for the pho...,booth,small enclosed compartment for phone calls or ...,small enclosed compartment for phone calls or ...,True


## Step 3 : Running the model on our own sentences

Each word below is placed in two different sentences, to check that the model really reads the
context instead of repeating whichever meaning is commonest.

The last pair is deliberately included as a failure. *form* meaning **shape** has only one example
sentence in the training data, so the model has almost nothing to associate with it and falls back
on the document meaning both times. This is the weakness noted earlier: rare senses stay hard no
matter which method is used.

In [9]:
tests = [('He hit the ball with his cricket bat.', 'bat'),
         ('A bat flew out of the cave at night.', 'bat'),
         ('The flowers bloom in spring every year.', 'spring'),
         ('The metal spring inside the clock is broken.', 'spring'),
         ('The dog began to bark at the stranger.', 'bark'),
         ('The bark of the old tree is rough.', 'bark'),
         ('Please fill this form and submit it.', 'form'),
         ('Water can take the form of ice or steam.', 'form')]

for sentence, word in tests:
    print(f'{word:7}| {sentence:45} -> {final_model(sentence, word)}')

bat    | He hit the ball with his cricket bat.         -> club used to hit the ball in sports (cricket, baseball) or as a weapon
bat    | A bat flew out of the cave at night.          -> nocturnal flying mammal
spring | The flowers bloom in spring every year.       -> season between winter and summer
spring | The metal spring inside the clock is broken.  -> coiled elastic metal device (or elasticity)
bark   | The dog began to bark at the stranger.        -> sound a dog makes
bark   | The bark of the old tree is rough.            -> tough outer covering of a tree
form   | Please fill this form and submit it.          -> document with blank spaces to fill in information
form   | Water can take the form of ice or steam.      -> document with blank spaces to fill in information


**What we found.** Lesk needs no training but only reaches about 0.60, because a one-line dictionary
definition rarely uses the same words as a real sentence. Learning the words that surround each sense
from the training sentences pushes this to about 0.71, and topping the rare senses up with WordNet
definitions takes it to about 0.73, against a lazy baseline of 0.51.

What holds the score back is the data, not the method. A sense has about four example sentences on
average, and the sentences are short, so for many senses there is very little context to learn from.
Senses with six or more examples are handled noticeably better than senses with one or two.